# Leksjon 11 - Model Context Protocol (MCP)

The **Model Context Protocol (MCP)** er en åpen standard som gjør det mulig for agenter å dynamisk oppdage og bruke verktøy, ressurser og datakilder ved kjøretid. I stedet for å hardkode verktøy inn i en agent, lar MCP agenter koble seg til eksterne servere som eksponerer funksjonalitet på forespørsel.

I denne leksjonen lærer du:
- Hva MCP er og hvorfor det er viktig for agentsystemer
- Hvordan MCP-klient-server-arkitekturen fungerer
- Hvordan bygge agenter som bruker verktøyoppdagelse i MCP-stil


## Oppsett

**Forutsetninger:**
- Azure AI Foundry-prosjekt med en utrullet modell
- Kjør `az login` for autentisering


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Hva er Model Context Protocol (MCP)?

MCP definerer en standard måte for AI-agenter å oppdage og samhandle med eksterne verktøy og datakilder:

- **MCP-server**: Eksponerer verktøy, ressurser og forespørsler via en standard protokoll
- **MCP-klient**: Agent-runtime som kobler seg til servere og oppdager tilgjengelig funksjonalitet
- **Dynamisk oppdagelse**: Agenter trenger ikke hardkodede verktøy — de oppdager hva som er tilgjengelig ved kjøretid

Dette er nyttig for å bygge utvidbare agentsystemer der nye funksjoner kan legges til uten å endre agentkoden.


## Hvordan MCP fungerer

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agenten (MCP-klient) kobler til en MCP-server
2. Serveren svarer med en liste over tilgjengelige verktøy og deres skjemaer
3. Agenten kan deretter kalle ethvert oppdaget verktøy mens den resonerer
4. Resultatene sendes tilbake gjennom samme protokoll


## Simulering av MCP-verktøyoppdagelse

Siden en ekte MCP-server krever en kjørende serverprosess, vil vi demonstrere mønsteret ved å bruke `@tool`-funksjoner som simulerer hva en MCP-tilknyttet innkvarteringstjeneste ville tilby.

I produksjon vil disse verktøyene bli oppdaget dynamisk fra en MCP-server i stedet for å være definert lokalt.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Bygge en agent med MCP-stilverktøy


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP i produksjon

I et produksjonsmiljø muliggjør MCP kraftige mønstre:

- **Dynamisk verktøyoppdagelse**: Agenter kobler seg til MCP-servere og oppdager verktøy under kjøring
- **Løst koblet arkitektur**: Verktøyleverandører kan oppdatere uavhengig av agenten
- **Deling på tvers av organisasjoner**: Team kan eksponere funksjonalitet via MCP-servere som enhver agent kan bruke
- **Microsoft Agent Framework-støtte**: MAF har innebygd MCP-klientstøtte via `mcp`-integrasjonen

For å bruke en ekte MCP-server med MAF, kobler du til via `hosted_mcp_tool()` eller MCP-klientintegrasjonen.

**Lær mer:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Sammendrag

I denne leksjonen lærte du:
- **MCP** er en åpen standard for dynamisk verktøyoppdagelse mellom agenter og verktøyleverandører
- Den **klient-server-arkitekturen** lar agenter oppdage egenskaper ved kjøretid
- MCP muliggjør **utvidbare, løst koblede agentsystemer** hvor verktøy kan legges til uten kodeendringer
- Microsoft Agent Framework tilbyr **innebygd MCP-støtte** for produksjonsbruk


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Ansvarsfraskrivelse:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten Co-op Translator (https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vennligst vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket bør anses som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår som følge av bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
